# Amparo -- Baseline y fine-tuning LoRA del agente legal

Este notebook:
1. Carga `dataset_legal.jsonl` (1320 ejemplos en 24 categorias) y separa una validacion estratificada por categoria que **nunca** se usa para entrenar.
2. Corre el modelo base sin modificar sobre esa validacion (**baseline**).
3. Entrena un adaptador **LoRA** sobre el resto del dataset.
4. Vuelve a correr el modelo ya afinado sobre la misma validacion.
5. Compara baseline vs. afinado -- esa comparacion es la evidencia real de que el fine-tuning mejoro algo, no solo una suposicion.

Antes de correr: `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU` (una T4 gratis alcanza usando 4-bit/QLoRA).

**Nota sobre las librerias:** `transformers`, `peft` y `trl` cambian su API con cierta frecuencia. Este notebook esta escrito contra las versiones estables mas recientes conocidas al momento de escribirlo; si algun cambio de firma da error, revisa el changelog de la libreria correspondiente -- la logica de cada celda (que hace y por que) sigue siendo valida aunque cambie algun nombre de parametro.

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets wandb
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 55.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-

## Configuracion

Cambia `MODEL_ID` para comparar Qwen2.5 7B vs. Llama 3.1 8B (o el que haya ganado en el comparador local `tools/model_comparator`).

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # alternativa: "meta-llama/Llama-3.1-8B-Instruct"

VAL_FRACTION = 0.15   # % por categoria reservado para validacion (nunca se entrena con esto)
RANDOM_SEED = 42

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8   # batch efectivo = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS
MAX_SEQ_LENGTH = 1024

MAX_NEW_TOKENS_EVAL = 300
OUTPUT_DIR = "/content/amparo-lora"

## Monta tu Google Drive y ubica el dataset

Sube `dataset_legal.jsonl` una sola vez a `MyDrive/Colab Notebooks/Amparo/dataset_legal.jsonl` (ajusta `DATASET_PATH` si usaste otra ruta). Con Drive no tienes que volver a subir el archivo cada vez que el entorno de Colab se reinicia.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DATASET_PATH = '/content/drive/MyDrive/Colab Notebooks/Amparo/dataset_legal.jsonl'  # ajusta si usaste otra ruta
print(f'Usando dataset: {DATASET_PATH}')

Mounted at /content/drive
Usando dataset: /content/drive/MyDrive/Colab Notebooks/Amparo/dataset_legal.jsonl


## Cargar y separar train / validacion (estratificado por categoria)

In [ ]:
import json
import random
from collections import defaultdict

random.seed(RANDOM_SEED)

records = [json.loads(line) for line in open(DATASET_PATH, encoding='utf-8')]

by_category = defaultdict(list)
for r in records:
    by_category[r['category']].append(r)

train_records, val_records = [], []
for category, items in by_category.items():
    items = items[:]
    random.shuffle(items)
    n_val = max(1, round(len(items) * VAL_FRACTION))
    val_records.extend(items[:n_val])
    train_records.extend(items[n_val:])

random.shuffle(train_records)
random.shuffle(val_records)

print(f'Total: {len(records)} | Train: {len(train_records)} | Validacion: {len(val_records)}')
for category in sorted(by_category):
    n_val = sum(1 for r in val_records if r['category'] == category)
    n_train = sum(1 for r in train_records if r['category'] == category)
    print(f'  {category:45s} train={n_train:3d}  val={n_val:3d}')

Total: 1320 | Train: 1119 | Validacion: 201
  Acceso a informacion publica                  train= 43  val=  8
  Accidentes de transito                        train= 52  val=  9
  Arriendo                                      train= 52  val=  9
  Comparendos de transito                       train= 52  val=  9
  Conciliacion prejudicial                      train= 43  val=  8
  Contratacion estatal y facturacion            train= 43  val=  8
  Contratos empresariales (B2B)                 train= 43  val=  8
  Derecho administrativo general                train= 43  val=  8
  Derecho ambiental sancionatorio               train= 43  val=  8
  Derecho contractual general                   train= 43  val=  8
  Derecho de familia - alimentos                train= 43  val=  8
  Despido                                       train= 53  val=  9
  Educacion / debido proceso disciplinario      train= 43  val=  8
  Embargos                                      train= 52  val=  9
  Garantias de con

## Cargar el modelo base en 4-bit (QLoRA)

Se carga cuantizado en 4-bit para que quepa comodamente en una GPU gratuita de Colab (T4, ~16GB).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

## Metrica de similitud

Misma heuristica lexica (difflib) que usa el comparador local en `tools/model_comparator/metrics.py`, para que los resultados sean comparables entre ambas herramientas. No es una metrica juridica rigurosa de correccion legal, solo sirve para ordenar/comparar respuestas de forma consistente.

In [ ]:
from difflib import SequenceMatcher

SYSTEM_PROMPT = records[0]['messages'][0]['content']


def similarity_pct(expected: str, actual: str) -> float:
    if not expected or not actual:
        return 0.0
    ratio = SequenceMatcher(None, expected.strip().lower(), actual.strip().lower()).ratio()
    return round(ratio * 100, 1)


@torch.no_grad()
def generate_response(model, query: str) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': query},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS_EVAL,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def evaluate(model, val_set, label: str):
    results = []
    for i, item in enumerate(val_set):
        query = item['messages'][1]['content']
        expected = item['messages'][2]['content']
        actual = generate_response(model, query)
        sim = similarity_pct(expected, actual)
        results.append({**item, 'generated': actual, 'similarity': sim})
        print(f"[{label}] {i + 1}/{len(val_set)}  sim={sim:5.1f}  {item['category']}")
    avg = sum(r['similarity'] for r in results) / len(results)
    print(f'\n[{label}] Similitud promedio: {avg:.1f}')
    return results

## Paso 1 -- Baseline (modelo base, sin fine-tuning)

Esto puede tardar varios minutos segun el tamano de la validacion.

In [ ]:
baseline_results = evaluate(model, val_records, label='baseline')

with open('/content/baseline_results.jsonl', 'w', encoding='utf-8') as f:
    for r in baseline_results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

[baseline] 1/201  sim=  3.1  Derecho de familia - alimentos
[baseline] 2/201  sim=  2.0  Propiedad y linderos
[baseline] 3/201  sim=  2.7  Salud / EPS
[baseline] 4/201  sim=  2.1  Educacion / debido proceso disciplinario
[baseline] 5/201  sim=  4.3  Accidentes de transito
[baseline] 6/201  sim=  2.8  Garantias de consumo
[baseline] 7/201  sim=  2.6  Relaciones laborales
[baseline] 8/201  sim=  3.6  Salud / EPS
[baseline] 9/201  sim= 13.3  Licencias urbanisticas
[baseline] 10/201  sim=  3.2  Conciliacion prejudicial
[baseline] 11/201  sim=  3.9  Derecho administrativo general
[baseline] 12/201  sim=  5.9  Licencias urbanisticas
[baseline] 13/201  sim=  3.4  Despido
[baseline] 14/201  sim=  5.6  Educacion / debido proceso disciplinario
[baseline] 15/201  sim=  2.9  Relaciones laborales
[baseline] 16/201  sim=  2.3  Derecho administrativo general
[baseline] 17/201  sim=  4.7  Salud / EPS
[baseline] 18/201  sim=  7.9  Contratos empresariales (B2B)
[baseline] 19/201  sim=  3.9  Pensiones y 

## Paso 2 -- Fine-tuning con LoRA

### Antes de entrenar: conecta Weights & Biases

M1 pide dejar registrada la curva de perdida del entrenamiento como evidencia visual de que el modelo esta aprendiendo. Crea una cuenta gratuita en wandb.ai si no tienes una, y ten a mano tu API key (la consigues en https://wandb.ai/authorize).

In [ ]:
import wandb

wandb.login()  # pega tu API key cuando te la pida (solo la primera vez)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tomasposada67 (tomasposada67-universidad-eafit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


In [ ]:
from datasets import Dataset


def format_for_training(record):
    return {'text': tokenizer.apply_chat_template(record['messages'], tokenize=False)}


train_dataset = Dataset.from_list(train_records).map(format_for_training)

Map:   0%|          | 0/1119 [00:00<?, ? examples/s]

In [ ]:
wandb.init(
    project='amparo-legal-finetune',
    name=f"lora-{MODEL_ID.split('/')[-1]}",
    config={
        'model_id': MODEL_ID,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
    },
)

from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_strategy='epoch',
    bf16=True,
    dataset_text_field='text',
    max_length=MAX_SEQ_LENGTH,
    report_to='wandb',
    run_name=f"lora-{MODEL_ID.split('/')[-1]}",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()

print('Dashboard de W&B (guarda este link para tu informe de M1):', wandb.run.url)
wandb.finish()

Tokenizing train dataset:   0%|          | 0/1119 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1119 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1119 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1119 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.431511
20,1.146347
30,0.944125
40,0.905491
50,0.873870
60,0.846638
70,0.859444
80,0.791507
90,0.775163
100,0.800980


Dashboard de W&B (guarda este link para tu informe de M1): https://wandb.ai/tomasposada67-universidad-eafit/amparo-legal-finetune/runs/nbxirzin


train/entropy,█▅▃▃▃▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁
train/epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▆▆▆▇▇▇███
train/global_step,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇███
train/grad_norm,█▁▂▁▁▁▂▂▂▂▂▂▂▃▂▃▂▃▂▃▃
train/learning_rate,██▇▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▁▁
train/loss,█▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/mean_token_accuracy,▁▆▇▇▇▇▇▇▇▇███████████
train/num_tokens,▁▁▂▂▂▃▃▃▄▄▄▅▅▆▆▆▇▇▇██
total_flos,2.075352344980992e+16
train/entropy,0.72494
train/epoch,3


## Guardar el adaptador LoRA en Drive

Se guarda directo en tu Drive (no solo son unos MB, son los pesos LoRA) para que no se pierda si el entorno de Colab se desconecta a mitad del entrenamiento o despues.

In [ ]:
import shutil

model.save_pretrained(f'{OUTPUT_DIR}/adapter')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/adapter')

DRIVE_ADAPTER_DIR = '/content/drive/MyDrive/Colab Notebooks/Amparo/amparo-lora-adapter'
shutil.copytree(f'{OUTPUT_DIR}/adapter', DRIVE_ADAPTER_DIR, dirs_exist_ok=True)
print(f'Adaptador guardado en {DRIVE_ADAPTER_DIR}')

Adaptador guardado en /content/drive/MyDrive/Colab Notebooks/Amparo/amparo-lora-adapter


In [ ]:
# prepare_model_for_kbit_training() dejo gradient checkpointing activo, lo que fuerza use_cache=False
# durante model.generate() (recalcula toda la atencion en cada token, sin KV-cache). Sin esto, el
# Paso 3 tarda varias veces mas de lo necesario y puede agotar la cuota de GPU de Colab a mitad de la evaluacion.
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

## Paso 3 -- Evaluar el modelo ya afinado (misma validacion)

In [ ]:
finetuned_results = evaluate(model, val_records, label='fine-tuned')

with open('/content/finetuned_results.jsonl', 'w', encoding='utf-8') as f:
    for r in finetuned_results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

[fine-tuned] 1/201  sim=  2.4  Derecho de familia - alimentos
[fine-tuned] 2/201  sim= 11.2  Propiedad y linderos
[fine-tuned] 3/201  sim=  5.1  Salud / EPS
[fine-tuned] 4/201  sim= 30.0  Educacion / debido proceso disciplinario
[fine-tuned] 5/201  sim= 18.9  Accidentes de transito
[fine-tuned] 6/201  sim= 30.7  Garantias de consumo
[fine-tuned] 7/201  sim= 59.2  Relaciones laborales
[fine-tuned] 8/201  sim= 52.3  Salud / EPS
[fine-tuned] 9/201  sim= 31.4  Licencias urbanisticas
[fine-tuned] 10/201  sim=  3.7  Conciliacion prejudicial
[fine-tuned] 11/201  sim=  7.2  Derecho administrativo general
[fine-tuned] 12/201  sim=  7.3  Licencias urbanisticas
[fine-tuned] 13/201  sim=  6.9  Despido
[fine-tuned] 14/201  sim=  5.4  Educacion / debido proceso disciplinario
[fine-tuned] 15/201  sim= 25.4  Relaciones laborales
[fine-tuned] 16/201  sim= 54.1  Derecho administrativo general
[fine-tuned] 17/201  sim=  2.5  Salud / EPS
[fine-tuned] 18/201  sim= 20.4  Contratos empresariales (B2B)
[fine-

## Paso 4 -- Comparacion baseline vs. afinado

Esta tabla es la evidencia real (no una suposicion) de si el fine-tuning mejoro las respuestas, por categoria y en promedio general.

In [ ]:
import pandas as pd

df_base = pd.DataFrame(baseline_results)[['id', 'category', 'similarity']].rename(columns={'similarity': 'baseline'})
df_ft = pd.DataFrame(finetuned_results)[['id', 'similarity']].rename(columns={'similarity': 'fine_tuned'})
comparison = df_base.merge(df_ft, on='id')
comparison['mejora'] = comparison['fine_tuned'] - comparison['baseline']

summary = comparison.groupby('category')[['baseline', 'fine_tuned', 'mejora']].mean().round(1)
print(summary)
print(
    f"\nPromedio general -> baseline: {comparison['baseline'].mean():.1f}  "
    f"fine-tuned: {comparison['fine_tuned'].mean():.1f}  "
    f"mejora: {comparison['mejora'].mean():+.1f}"
)

comparison.to_csv('/content/comparacion_baseline_vs_finetuned.csv', index=False)

                                          baseline  fine_tuned  mejora
category                                                              
Acceso a informacion publica                   2.6         7.2     4.5
Accidentes de transito                         3.6        18.4    14.9
Arriendo                                       3.0        12.2     9.2
Comparendos de transito                        3.2        30.4    27.2
Conciliacion prejudicial                       5.5        12.3     6.8
Contratacion estatal y facturacion             2.4        18.9    16.5
Contratos empresariales (B2B)                  4.7        20.8    16.0
Derecho administrativo general                 3.3        16.4    13.1
Derecho ambiental sancionatorio                3.0        28.9    25.9
Derecho contractual general                    3.3        14.7    11.4
Derecho de familia - alimentos                 4.0         8.8     4.8
Despido                                        3.2        12.6     9.4
Educac

## Modo rapido -- Probar el modelo ya entrenado sin correr el resto del notebook

Para probar el modelo (hoy, otro dia, o tras una desconexion) sin gastar tiempo/cuota de GPU repitiendo el Paso 1 a Paso 4, corre solo:

1. La primera celda del notebook (instalacion de dependencias) -- Colab reinicia el runtime sin paquetes instalados cada vez.
2. La celda de abajo: monta Drive, carga el modelo base + el adaptador LoRA que ya guardaste, y deja todo listo.

y despues ve directo al Paso 5. Corre esta celda incluso si acabas de entrenar en esta misma sesion -- asegura que generate_response y el adaptador cargado coincidan con lo que quedo guardado en Drive.

In [ ]:
import json
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

drive.mount('/content/drive')

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MAX_NEW_TOKENS_EVAL = 300
DATASET_PATH = '/content/drive/MyDrive/Colab Notebooks/Amparo/dataset_legal.jsonl'
DRIVE_ADAPTER_DIR = '/content/drive/MyDrive/Colab Notebooks/Amparo/amparo-lora-adapter'

with open(DATASET_PATH, encoding='utf-8') as f:
    SYSTEM_PROMPT = json.loads(f.readline())['messages'][0]['content']

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model = PeftModel.from_pretrained(model, DRIVE_ADAPTER_DIR)
model.eval()


@torch.no_grad()
def generate_response(model, query: str) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': query},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS_EVAL,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


print(f'Modelo + adaptador cargados desde {DRIVE_ADAPTER_DIR}. Listo para el Paso 5 -- no hizo falta correr nada mas del notebook.')

## Paso 5 -- Probar el modelo afinado con tus propias preguntas

Cuadro de texto interactivo: escribe una consulta legal, obten la respuesta del modelo ya afinado, y sigue probando las veces que quieras. Necesita model y generate_response -- ya definidos si corriste el notebook completo (Metrica de similitud, Paso 1 a Paso 4), o si corriste "Modo rapido" de arriba.

Cada pregunta y respuesta se guarda en paso5_pruebas.jsonl dentro de tu Drive, asi que el historial de pruebas *no se pierde* al cerrarse la sesion de Colab: la proxima vez que retomes esta prueba, las nuevas pruebas se agregan al mismo archivo en vez de reemplazarlo.

In [ ]:
import datetime
import ipywidgets as widgets
from IPython.display import display

PASO5_LOG_PATH = '/content/drive/MyDrive/Colab Notebooks/Amparo/paso5_pruebas.jsonl'

pregunta_box = widgets.Text(
    placeholder='Escribe tu consulta legal y presiona Enter o el boton...',
    layout=widgets.Layout(width='100%'),
)
enviar_btn = widgets.Button(description='Preguntar', button_style='primary')
salida = widgets.Output()
estado = {'ocupado': False}


def preguntar(_):
    if estado['ocupado']:
        return
    pregunta = pregunta_box.value.strip()
    if not pregunta:
        return
    estado['ocupado'] = True
    pregunta_box.value = ''
    pregunta_box.disabled = True
    enviar_btn.disabled = True
    enviar_btn.description = 'Generando...'

    try:
        respuesta = generate_response(model, pregunta)

        with salida:
            print(f'> {pregunta}')
            print(respuesta)
            print('-' * 80)

        with open(PASO5_LOG_PATH, 'a', encoding='utf-8') as log_file:
            log_file.write(json.dumps({
                'timestamp': datetime.datetime.now().isoformat(timespec='seconds'),
                'pregunta': pregunta,
                'respuesta': respuesta,
            }, ensure_ascii=False) + '\n')
    finally:
        pregunta_box.disabled = False
        enviar_btn.disabled = False
        enviar_btn.description = 'Preguntar'
        estado['ocupado'] = False


enviar_btn.on_click(preguntar)
pregunta_box.on_submit(preguntar)

display(widgets.HBox([pregunta_box, enviar_btn]), salida)

Pregunta (Enter vacio para salir): cOMO LIQUIDAN LOS IMPUESTOS DE LOS VEHICULOS
Los vehiculos suelen tener impuestos propios (como el IVA) que se liquidan al momento de la compra o cambio de titularidad. Verifica con la entidad correspondiente (generalmente la Dian) los requisitos especificos para cada tipo de vehiculo.
--------------------------------------------------------------------------------
Pregunta (Enter vacio para salir): Me pueden echar del trabajo sin darme explicaciones ni notificarme?
El despido debe tener una causa especifica y formalmente notificado. Si no te han dado ninguna explicacion o notificacion, puedes solicitar esa informacion para evaluar tu situacion.
--------------------------------------------------------------------------------
Pregunta (Enter vacio para salir): mi EPS puede negarse a prestarme servicio?
Las EPS tienen obligaciones de prestacion y cobertura especificas. Si consideras que tu situacion no esta siendo atendida correctamente, puedes presenta

KeyboardInterrupt: Interrupted by user

### (Opcional) Ver el historial acumulado de pruebas

Lee paso5_pruebas.jsonl desde Drive -- funciona aunque el modelo no este cargado en esta sesion, porque solo lee el archivo de log, no depende de model ni generate_response.

In [ ]:
import pandas as pd

PASO5_LOG_PATH = '/content/drive/MyDrive/Colab Notebooks/Amparo/paso5_pruebas.jsonl'
pd.read_json(PASO5_LOG_PATH, lines=True).tail(20)